# Binary and Specialist Formats

Not everything is CSV or JSON. Satellite teams love binary formats for speed. International agencies use XML for interchange. DevOps uses YAML for configuration. A data engineer needs to speak all these languages.

In the previous notebooks we covered the workhorses -- CSV, JSON, and Parquet. Those three will handle perhaps 80% of your data format encounters. This notebook is about the other 20%: the specialist formats that appear in specific domains and for specific purposes.

We will look at three:

- **YAML** -- the format that powers configuration everywhere from Kubernetes to GitHub Actions
- **XML** -- the granddaddy of structured data exchange, still very much alive in government and scientific data
- **MessagePack** -- a binary serialisation format that is like JSON but faster and smaller

We will finish by building a decision matrix: given a new data source, how do you choose the right format?

## Setup

In [ ]:
%pip install -q pyyaml lxml msgpack

In [ ]:
import yaml
from lxml import etree
import msgpack
import json
import pandas as pd
import time

## Part 1: YAML -- Configuration, Not Data

YAML ("YAML Ain't Markup Language") is a human-readable data serialisation format. If you have used Docker Compose, GitHub Actions, Ansible, or Kubernetes, you have already written YAML.

It is excellent for configuration files because it is clean, readable, and supports comments (which JSON does not). But it is a poor choice for storing data -- it is slow to parse and not designed for tabular structures.

Let's read a YAML configuration file for our weather station network.

In [ ]:
with open("../data/station_config.yaml", "r") as f:
    config = yaml.safe_load(f)

print(type(config))
print(list(config.keys()))

YAML maps to Python dictionaries and lists, just like JSON. The `safe_load` function is important -- always use it instead of `yaml.load()`, which can execute arbitrary Python code embedded in the YAML file. That is a genuine security risk.

In [ ]:
# Explore the network configuration
print(f"Network name: {config['network']['name']}")
print(f"Version: {config['network']['version']}")
print(f"Contact: {config['network']['contact']['email']}")

In [ ]:
# Explore the station definitions
for station in config["stations"]:
    sensors = ", ".join(station["sensors"])
    print(f"{station['id']} - {station['name']} ({station['country']}): [{sensors}] - {station['status']}")

In [ ]:
# Default settings
print("Default settings:")
for key, value in config["defaults"].items():
    print(f"  {key}: {value}")

In [ ]:
# Alert thresholds
print("Alert thresholds:")
alerts = config["alerts"]
print(f"  Temperature range: {alerts['temperature_range']['min_c']}C to {alerts['temperature_range']['max_c']}C")
print(f"  Max wind speed: {alerts['wind_speed_max_ms']} m/s")
print(f"  Pressure range: {alerts['pressure_range']['min_hpa']} to {alerts['pressure_range']['max_hpa']} hPa")

### Writing YAML

You can also write Python objects back to YAML. This is useful when generating configuration files programmatically.

In [ ]:
# Create a new station config and convert to YAML
new_station = {
    "id": "WS050",
    "name": "Edinburgh Royal Mile",
    "country": "United Kingdom",
    "location": {"lat": 55.9502, "lon": -3.1875, "elevation_m": 80},
    "sensors": ["temperature", "humidity", "pressure", "rainfall"],
    "calibration_date": "2024-06-01",
    "status": "active",
}

print(yaml.dump(new_station, default_flow_style=False, sort_keys=False))

### When to use YAML

YAML is for **configuration**, not for **data**. Use it when:
- Humans need to read and edit the file regularly
- You need comments in the file
- The structure is hierarchical but small
- You are configuring an application or pipeline

Do not use YAML for storing datasets, logs, or anything with thousands of records.

## Part 2: XML -- The Format That Refuses to Die

XML (eXtensible Markup Language) was the dominant data interchange format before JSON took over. It is verbose, heavy, and frankly painful to work with. So why are we learning it?

Because it is everywhere in certain domains:
- Government open data (HMRC, NHS, EU agencies)
- Scientific data catalogues (NASA, ESA, NOAA)
- Financial services (FIX, SWIFT messages)
- Legacy enterprise systems (SOAP APIs)
- Office documents (DOCX, XLSX are XML inside a ZIP file)

As a data engineer, you will encounter XML whether you like it or not. Let's learn to parse it efficiently.

### Reading XML with lxml

In [ ]:
tree = etree.parse("../data/satellite_metadata.xml")
root = tree.getroot()

print(f"Root tag: {root.tag}")
print(f"Root attributes: {dict(root.attrib)}")
print(f"Number of child elements: {len(root)}")

XML is a tree structure. The root element is `<catalogue>`, and it contains `<dataset>` children. Each dataset has sub-elements for name, satellite, instrument, and so on.

### Navigating the tree

In [ ]:
# Look at the first dataset
first = root[0]
print(f"Tag: {first.tag}")
print(f"Attributes: {dict(first.attrib)}")

for child in first:
    if len(child) == 0:
        print(f"  {child.tag}: {child.text}")
    else:
        print(f"  {child.tag}: [{len(child)} children]")

### XPath -- querying XML

XPath is to XML what SQL is to databases. It is a query language for selecting parts of an XML document. Here are the most useful patterns.

In [ ]:
# Find all dataset names
names = root.xpath("//dataset/name/text()")
for name in names:
    print(name)

In [ ]:
# Find datasets from a specific satellite
terra_datasets = root.xpath("//dataset[satellite='Terra']")
print(f"Terra datasets: {len(terra_datasets)}")
for ds in terra_datasets:
    print(f"  {ds.find('name').text} ({ds.find('instrument').text})")

In [ ]:
# Find datasets with resolution under 1 km
high_res = root.xpath("//dataset[resolution_km < 1]")
print(f"High-resolution datasets (< 1 km): {len(high_res)}")
for ds in high_res:
    print(f"  {ds.find('name').text}: {ds.find('resolution_km').text} km")

### XPath cheat sheet

| Expression | Meaning |
|---|---|
| `//dataset` | All `<dataset>` elements anywhere |
| `//dataset/name` | All `<name>` children of `<dataset>` |
| `//dataset/name/text()` | The text content of those `<name>` elements |
| `//dataset[@id='SAT-001']` | `<dataset>` with attribute `id` equal to `SAT-001` |
| `//dataset[satellite='Terra']` | `<dataset>` where child `<satellite>` text is `Terra` |

### Converting XML to a DataFrame

For analysis, we usually want to get XML data into a DataFrame. We need to extract the relevant fields from each element.

In [ ]:
rows = []
for dataset in root.xpath("//dataset"):
    row = {
        "dataset_id": dataset.get("id"),
        "name": dataset.findtext("name"),
        "satellite": dataset.findtext("satellite"),
        "instrument": dataset.findtext("instrument"),
        "resolution_km": float(dataset.findtext("resolution_km")),
        "start_date": dataset.findtext("temporal_coverage/start"),
        "end_date": dataset.findtext("temporal_coverage/end"),
        "num_variables": len(dataset.xpath("variables/variable")),
    }
    # Also extract variable names
    var_names = [v.get("name") for v in dataset.xpath("variables/variable")]
    row["variables"] = ", ".join(var_names)
    rows.append(row)

df_satellites = pd.DataFrame(rows)
df_satellites

In [ ]:
# Quick analysis: datasets by satellite
df_satellites.groupby("satellite")["name"].count().sort_values(ascending=False)

In [ ]:
# Resolution distribution
df_satellites[["name", "resolution_km"]].sort_values("resolution_km")

### pandas `read_xml()` shortcut

For simple XML structures, pandas has a built-in `read_xml()` function. It works well for flat structures but struggles with deeply nested XML.

In [ ]:
# This gets the top-level fields but misses nested ones
df_simple_xml = pd.read_xml("../data/satellite_metadata.xml", xpath="//dataset")
df_simple_xml.head()

For anything beyond flat structures, the manual lxml approach we used above gives you much more control.

## Part 3: MessagePack -- Binary JSON

MessagePack is a binary serialisation format. Think of it as JSON, but stored in binary instead of text. This makes it:
- **Smaller**: numbers are stored as actual numbers, not strings of digits
- **Faster**: no text parsing needed
- **Type-preserving**: integers stay integers, floats stay floats

The trade-off: it is not human-readable. You cannot open a MessagePack file in a text editor.

MessagePack is popular for:
- Inter-service communication (microservices talking to each other)
- Caching (Redis can store MessagePack natively)
- IoT and embedded systems (where bandwidth is precious)
- Any situation where you want JSON's flexibility with better performance

In [ ]:
# Create some sample data -- a batch of sensor readings
sensor_data = [
    {"station_id": "WS001", "timestamp": "2024-03-15T10:00:00Z", "temp_c": 12.5, "humidity_pct": 65.2, "pressure_hpa": 1013.2},
    {"station_id": "WS001", "timestamp": "2024-03-15T11:00:00Z", "temp_c": 13.1, "humidity_pct": 63.8, "pressure_hpa": 1012.8},
    {"station_id": "WS001", "timestamp": "2024-03-15T12:00:00Z", "temp_c": 14.7, "humidity_pct": 58.1, "pressure_hpa": 1012.5},
    {"station_id": "WS001", "timestamp": "2024-03-15T13:00:00Z", "temp_c": 15.2, "humidity_pct": 55.6, "pressure_hpa": 1012.1},
    {"station_id": "WS001", "timestamp": "2024-03-15T14:00:00Z", "temp_c": 15.8, "humidity_pct": 54.3, "pressure_hpa": 1011.9},
]

print(f"Number of readings: {len(sensor_data)}")

In [ ]:
# Serialise to MessagePack
packed = msgpack.packb(sensor_data)
print(f"Type: {type(packed)}")
print(f"Size: {len(packed)} bytes")

# Compare with JSON
json_bytes = json.dumps(sensor_data).encode("utf-8")
print(f"JSON size: {len(json_bytes)} bytes")
print(f"MessagePack is {len(json_bytes) / len(packed):.1f}x smaller")

In [ ]:
# Deserialise back
unpacked = msgpack.unpackb(packed)
print(type(unpacked))
print(unpacked[0])

### Speed comparison

Let's create a larger dataset and compare serialisation/deserialisation speeds.

In [ ]:
import random

# Generate a larger dataset for benchmarking
large_data = []
for i in range(10000):
    large_data.append({
        "station_id": f"WS{i % 25 + 1:03d}",
        "timestamp": f"2024-03-15T{i % 24:02d}:00:00Z",
        "temp_c": round(random.uniform(-10, 35), 1),
        "humidity_pct": round(random.uniform(20, 100), 1),
        "pressure_hpa": round(random.uniform(980, 1040), 1),
    })

print(f"Records: {len(large_data)}")

In [ ]:
# JSON serialisation
start = time.time()
json_bytes = json.dumps(large_data).encode("utf-8")
json_pack_time = time.time() - start

start = time.time()
json.loads(json_bytes.decode("utf-8"))
json_unpack_time = time.time() - start

# MessagePack serialisation
start = time.time()
msgpack_bytes = msgpack.packb(large_data)
msgpack_pack_time = time.time() - start

start = time.time()
msgpack.unpackb(msgpack_bytes)
msgpack_unpack_time = time.time() - start

print(f"{'':>20} {'JSON':>10} {'MsgPack':>10} {'Ratio':>10}")
print(f"{'Size (bytes)':>20} {len(json_bytes):>10,} {len(msgpack_bytes):>10,} {len(json_bytes)/len(msgpack_bytes):>10.1f}x")
print(f"{'Pack time (ms)':>20} {json_pack_time*1000:>10.2f} {msgpack_pack_time*1000:>10.2f} {json_pack_time/msgpack_pack_time:>10.1f}x")
print(f"{'Unpack time (ms)':>20} {json_unpack_time*1000:>10.2f} {msgpack_unpack_time*1000:>10.2f} {json_unpack_time/msgpack_unpack_time:>10.1f}x")

MessagePack should be both smaller and faster than JSON. The exact ratios depend on the data, but 1.5-2x improvements are typical.

### Writing and reading MessagePack files

In [ ]:
# Write to a file
with open("/tmp/sensor_data.msgpack", "wb") as f:
    msgpack.pack(large_data, f)

# Read from a file
with open("/tmp/sensor_data.msgpack", "rb") as f:
    loaded = msgpack.unpack(f)

print(f"Loaded {len(loaded)} records")
print(loaded[0])

Notice the API mirrors JSON: `pack`/`unpack` for files, `packb`/`unpackb` for bytes. If you know the JSON API, you already know the MessagePack API.

## Part 4: The Format Decision Matrix

We have now covered six data formats across three notebooks. How do you choose the right one for a given situation? Here is a decision matrix.

In [ ]:
matrix = pd.DataFrame({
    "Format": ["CSV", "JSON", "Parquet", "YAML", "XML", "MessagePack"],
    "Human-readable?": ["Yes", "Yes", "No", "Yes", "Yes (verbose)", "No"],
    "Schema?": ["No", "No (JSON Schema optional)", "Yes (embedded)", "No", "Yes (XSD optional)", "No"],
    "Columnar?": ["No", "No", "Yes", "No", "No", "No"],
    "Compressed?": ["No", "No", "Yes (built-in)", "No", "No", "Partially"],
    "Best for": [
        "Small tabular data, interchange",
        "APIs, nested/hierarchical data",
        "Large analytical datasets",
        "Configuration files",
        "Government/scientific interchange",
        "Fast binary serialisation",
    ],
})
matrix

### The decision flowchart

When you encounter a new data source or need to choose a format, walk through these questions:

**1. Is it configuration or data?**
- Configuration -> YAML (or JSON if you do not need comments)
- Data -> continue

**2. Is it tabular (rows and columns)?**
- Yes, and small (< few thousand rows) -> CSV
- Yes, and large (millions of rows or wide tables) -> Parquet
- No, it is hierarchical/nested -> continue

**3. Does it need to be human-readable?**
- Yes -> JSON (or XML if required by the receiving system)
- No -> continue

**4. Is size or speed critical?**
- Yes -> MessagePack (or Parquet if it is tabular)
- No -> JSON is probably fine

**5. Does the receiving system mandate a format?**
- If a government API requires XML, you send XML.
- If a legacy system only reads CSV, you send CSV.
- Pragmatism beats preference.

### Applying the flowchart

Let's practise. Consider these scenarios:

**Scenario A:** You need to store 50 million rows of transaction data for analytical queries. Analysts typically query 3-4 columns at a time.

Answer: **Parquet**. Large, tabular, analytical use case, column pruning is essential.

**Scenario B:** A partner agency sends you daily weather forecasts in a structured format with nested regions and time periods.

Answer: **JSON** (or XML if they are a government agency). Hierarchical data, human-readable, standard interchange format.

**Scenario C:** You need to define the settings for a data pipeline -- database connection strings, retry policies, logging levels.

Answer: **YAML**. Configuration, not data. Human-readable, supports comments.

**Scenario D:** Two microservices need to exchange sensor readings at high frequency over a message queue.

Answer: **MessagePack**. Speed and size matter, human readability does not.

## Part 5: Bringing It All Together

Let's do a final side-by-side comparison. We will take the same data and serialise it in every format we have covered, comparing size and speed.

In [ ]:
# Use the buoy data from notebook 1 as our test dataset
with open("../data/ocean_buoys.json", "r") as f:
    buoy_data = json.load(f)

print(f"Records: {len(buoy_data)}")
print(f"Fields per record: {len(buoy_data[0])}")

In [ ]:
# JSON
start = time.time()
json_str = json.dumps(buoy_data)
json_time = time.time() - start
json_size = len(json_str.encode("utf-8"))

# MessagePack
start = time.time()
msgpack_data = msgpack.packb(buoy_data)
msgpack_time = time.time() - start
msgpack_size = len(msgpack_data)

# CSV (flatten first)
df_buoys = pd.json_normalize(buoy_data, sep="_")
start = time.time()
csv_str = df_buoys.to_csv(index=False)
csv_time = time.time() - start
csv_size = len(csv_str.encode("utf-8"))

# YAML
start = time.time()
yaml_str = yaml.dump(buoy_data, default_flow_style=False)
yaml_time = time.time() - start
yaml_size = len(yaml_str.encode("utf-8"))

results = pd.DataFrame({
    "Format": ["JSON", "MessagePack", "CSV", "YAML"],
    "Size (bytes)": [json_size, msgpack_size, csv_size, yaml_size],
    "Serialise time (ms)": [
        round(json_time * 1000, 2),
        round(msgpack_time * 1000, 2),
        round(csv_time * 1000, 2),
        round(yaml_time * 1000, 2),
    ],
})
results["Relative size"] = (results["Size (bytes)"] / results["Size (bytes)"].min()).round(1)
results

MessagePack will generally be the smallest and fastest. YAML is by far the slowest and most verbose -- which is fine, because you are not supposed to use it for data. CSV is compact for flat tabular data but cannot represent the nested structure without flattening.

There is no universally "best" format. Each one makes different trade-offs, and the right choice depends on your specific situation.

## Summary

| Format | Read with | Write with | Best for |
|---|---|---|---|
| YAML | `yaml.safe_load(f)` | `yaml.dump(obj)` | Configuration files |
| XML | `etree.parse(f)` + XPath | `etree.tostring()` | Government/scientific interchange |
| MessagePack | `msgpack.unpack(f)` | `msgpack.pack(obj, f)` | Fast binary serialisation |

And from the previous notebooks:

| Format | Read with | Write with | Best for |
|---|---|---|---|
| CSV | `pd.read_csv()` | `df.to_csv()` | Small tabular data, interchange |
| JSON | `json.load(f)` | `json.dump(obj, f)` | APIs, nested data |
| Parquet | `pd.read_parquet()` | `df.to_parquet()` | Large analytical datasets |

---

## Exercises

### Exercise 1: Parse the satellite catalogue

Using the satellite metadata XML file:

1. Find all datasets that use the MODIS instrument
2. For each, extract the dataset name, satellite, resolution, and the names of all variables
3. Put the results in a DataFrame

In [ ]:
# Your code here


### Exercise 2: YAML to DataFrame

Read the station configuration YAML file and convert the station list into a pandas DataFrame with columns: `id`, `name`, `country`, `lat`, `lon`, `elevation_m`, `num_sensors`, `status`.

In [ ]:
# Your code here


### Exercise 3: MessagePack round-trip

1. Load the ocean buoys JSON data
2. Serialise it to MessagePack and write to a file
3. Read it back from the file
4. Verify the data survived the round trip by checking the first record matches
5. Compare the file sizes of the JSON and MessagePack versions

In [ ]:
# Your code here


### Exercise 4: Format recommendation

For each of the following scenarios, recommend a format and explain your reasoning in a comment. Use the decision flowchart from Part 4.

1. A mobile weather app needs to send hourly readings from 100,000 devices to a central server. Bandwidth is limited.
2. The UK Met Office publishes a daily weather bulletin for public consumption.
3. Your team stores 2 billion rows of historical climate observations for research queries.
4. A European government agency requires you to submit environmental monitoring data in a standardised format with a formal schema definition.
5. You are setting up connection parameters and retry logic for a data pipeline that reads from three different APIs.

In [ ]:
# Write your recommendations as comments:

# Scenario 1:
#

# Scenario 2:
#

# Scenario 3:
#

# Scenario 4:
#

# Scenario 5:
#
